# Рекомендательная система на основе матричного разложения

**Задание:**
- Реализовать Latent Factor Model (SVD) с SGD и bias
- Сравнить с Most Popular, User‑CF, Item‑CF
- Проанализировать влияние размерности, регуляризации, числа эпох
- Метрики: RMSE, Precision@K, Recall@K, MAP@K, NDCG@K
- Доп. метрика: Coverage / Diversity
- Логирование в TensorBoard

In [23]:
!pip install pandas numpy matplotlib scikit-learn torch tensorboard requests tqdm ipywidgets

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 5.5 MB/s eta 0:00:01
   ---------------------- ----------------- 524.3/914.9 kB 5.5 MB/s eta 0:00:01
   ---------------------- ----------------- 524.3/914.9 kB 5.5 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 1.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 6.2 MB/s eta 0:00:01
   ---------------------------- ----------- 1.6/2.2 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 6.6 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidg

In [6]:
import os
import zipfile
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder

%matplotlib inline

## 1. Загрузка данных MovieLens 1M

In [7]:
url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
print("Скачивание MovieLens 1M...")
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall("ml-1m")

ratings = pd.read_csv(
    'ml-1m/ml-1m/ratings.dat',
    sep='::',
    names=['user_id', 'movie_id', 'rating', 'timestamp'],
    engine='python'
)
movies = pd.read_csv(
    'ml-1m/ml-1m/movies.dat',
    sep='::',
    names=['movie_id', 'title', 'genres'],
    engine='python',
    encoding='latin-1'
)

print(f"Рейтингов: {len(ratings)}, Пользователей: {ratings['user_id'].nunique()}, Фильмов: {ratings['movie_id'].nunique()}")
ratings.head()

Скачивание MovieLens 1M...
Рейтингов: 1000209, Пользователей: 6040, Фильмов: 3706


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## 2. Фильтрация данных для ускорения экспериментов

In [8]:
# Оставляем пользователей с ≥ 20 оценками
user_counts = ratings['user_id'].value_counts()
active_users = user_counts[user_counts >= 20].index
ratings_f = ratings[ratings['user_id'].isin(active_users)]

# Оставляем фильмы с ≥ 20 оценками
movie_counts = ratings_f['movie_id'].value_counts()
popular_movies = movie_counts[movie_counts >= 20].index
ratings_f = ratings_f[ratings_f['movie_id'].isin(popular_movies)]

# Переназначаем индексы с 0
ratings_f['user_idx'] = ratings_f['user_id'].astype('category').cat.codes
ratings_f['movie_idx'] = ratings_f['movie_id'].astype('category').cat.codes

num_users = ratings_f['user_idx'].nunique()
num_movies = ratings_f['movie_idx'].nunique()

print(f"После фильтрации: {len(ratings_f)} оценок, {num_users} пользователей, {num_movies} фильмов")
ratings_f.head()

После фильтрации: 995492 оценок, 6040 пользователей, 3043 фильмов


,user_id,movie_id,rating,timestamp,user_idx,movie_idx
0,1,1193,5,978300760,0,880
1,1,661,3,978302109,0,536
2,1,914,3,978301968,0,680
3,1,3408,4,978300275,0,2605
4,1,2355,5,978824291,0,1773


## 3. Разбиение на train / val / test

In [9]:
# Временное разбиение (по timestamp)
ratings_f = ratings_f.sort_values('timestamp')

train, test = train_test_split(ratings_f, test_size=0.2, random_state=42, stratify=ratings_f['user_idx'])
train, val = train_test_split(train, test_size=0.25, random_state=42, stratify=train['user_idx'])  # 0.25*0.8 = 0.2

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 597294, Val: 199099, Test: 199099


## 4. Базовые модели (Baselines)

In [15]:
# Most Popular – средний рейтинг фильма по обучающей выборке
movie_stats = train.groupby('movie_idx')['rating'].agg(['mean', 'count'])
# Отсекаем фильмы с малым числом оценок
C = train['rating'].mean()
m = 50  # порог
movie_stats['weighted_rating'] = (movie_stats['count'] / (movie_stats['count'] + m)) * movie_stats['mean'] + \
                                  (m / (movie_stats['count'] + m)) * C
top_popular = movie_stats.sort_values('weighted_rating', ascending=False).index.values

print("Топ-10 популярных фильмов (по взвешенному рейтингу):")
for mid in top_popular[:10]:
    title = movies[movies['movie_id'] == mid]['title'].values[0]
    print(f"{title} - {movie_stats.loc[mid, 'weighted_rating']:.2f}")

Топ-10 популярных фильмов (по взвешенному рейтингу):
Little Princess, A (1995) - 4.52
Seven (Se7en) (1995) - 4.48
With Honors (1994) - 4.48
Roula (1995) - 4.47
Dingo (1992) - 4.46
Sweet Nothing (1995) - 4.44
Kissed (1996) - 4.43
Dear Diary (Caro Diario) (1994) - 4.43
Browning Version, The (1994) - 4.41
World of Apu, The (Apur Sansar) (1959) - 4.41


In [16]:
def build_user_item_matrix(df, n_users, n_items):
    """Создаёт матрицу user×item с рейтингами, отсутствующие значения = 0."""
    mat = np.zeros((n_users, n_items))
    for row in df.itertuples():
        mat[row.user_idx, row.movie_idx] = row.rating
    return mat

def user_based_predict(user_id, item_id, train_matrix, user_sim_matrix, k=20):
    """Предсказание рейтинга методом UserKNN."""
    if item_id >= train_matrix.shape[1]:
        return np.mean(train_matrix[user_id][train_matrix[user_id] > 0])
    users_who_rated = np.where(train_matrix[:, item_id] > 0)[0]
    if len(users_who_rated) == 0:
        return np.mean(train_matrix[user_id][train_matrix[user_id] > 0])
    sims = user_sim_matrix[user_id, users_who_rated]
    if k < len(sims):
        top_k_idx = np.argsort(sims)[-k:]
        users_who_rated = users_who_rated[top_k_idx]
        sims = sims[top_k_idx]
    if np.sum(sims) == 0:
        return np.mean(train_matrix[user_id][train_matrix[user_id] > 0])
    ratings = train_matrix[users_who_rated, item_id]
    return np.dot(sims, ratings) / np.sum(sims)

def item_based_predict(user_id, item_id, train_matrix, item_sim_matrix, k=20):
    """Предсказание рейтинга методом ItemKNN."""
    if user_id >= train_matrix.shape[0]:
        return np.mean(train_matrix[:, item_id][train_matrix[:, item_id] > 0])
    items_rated_by_user = np.where(train_matrix[user_id, :] > 0)[0]
    if len(items_rated_by_user) == 0:
        return np.mean(train_matrix[:, item_id][train_matrix[:, item_id] > 0])
    sims = item_sim_matrix[item_id, items_rated_by_user]
    if k < len(sims):
        top_k_idx = np.argsort(sims)[-k:]
        items_rated_by_user = items_rated_by_user[top_k_idx]
        sims = sims[top_k_idx]
    if np.sum(sims) == 0:
        return np.mean(train_matrix[:, item_id][train_matrix[:, item_id] > 0])
    ratings = train_matrix[user_id, items_rated_by_user]
    return np.dot(sims, ratings) / np.sum(sims)

def evaluate_knn_model(predict_func, train_matrix, sim_matrix, eval_df, k=20):
    """Вычисляет RMSE на переданном DataFrame."""
    preds = []
    truths = []
    for row in eval_df.itertuples():
        pred = predict_func(row.user_idx, row.movie_idx, train_matrix, sim_matrix, k)
        preds.append(pred)
        truths.append(row.rating)
    return np.sqrt(np.mean((np.array(preds) - np.array(truths))**2))

In [18]:
# Построение матрицы обучающих данных
train_matrix = build_user_item_matrix(train, num_users, num_movies)
val_matrix = build_user_item_matrix(val, num_users, num_movies)

# Вычисление матриц сходства (если ещё не сделано)
if 'user_sim' not in globals():
    print("Вычисление матрицы сходства пользователей (косинус)...")
    user_sim = cosine_similarity(train_matrix)
if 'item_sim' not in globals():
    print("Вычисление матрицы сходства фильмов (косинус)...")
    item_sim = cosine_similarity(train_matrix.T)

# Оценка RMSE на валидационной выборке
rmse_ubcf = evaluate_knn_model(user_based_predict, train_matrix, user_sim, val, k=20)
rmse_ibcf = evaluate_knn_model(item_based_predict, train_matrix, item_sim, val, k=20)

print(f"User-based CF RMSE (val): {rmse_ubcf:.4f}")
print(f"Item-based CF RMSE (val): {rmse_ibcf:.4f}")

Вычисление матрицы сходства пользователей (косинус)...
Вычисление матрицы сходства фильмов (косинус)...
User-based CF RMSE (val): 0.9678
Item-based CF RMSE (val): 0.9398


## 5. Latent Factor Model (PyTorch)

In [19]:
class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.movies = torch.tensor(df['movie_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=50, add_bias=True):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, embedding_dim)
        self.item_emb = nn.Embedding(n_items, embedding_dim)
        self.add_bias = add_bias
        if add_bias:
            self.user_bias = nn.Embedding(n_users, 1)
            self.item_bias = nn.Embedding(n_items, 1)
            self.global_bias = nn.Parameter(torch.tensor(train['rating'].mean()))

        # Инициализация
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, u_ids, i_ids):
        u = self.user_emb(u_ids)
        i = self.item_emb(i_ids)
        dot = (u * i).sum(dim=1)
        if self.add_bias:
            dot += self.user_bias(u_ids).squeeze() + self.item_bias(i_ids).squeeze() + self.global_bias
        return torch.sigmoid(dot) * 4 + 1  # масштабирование в [1,5]


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for u, i, r in loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        preds = model(u, i)
        loss = criterion(preds, r)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_rmse(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)
            loss = criterion(preds, r)
            total_loss += loss.item()
    return np.sqrt(total_loss / len(loader))

## 6. Эксперименты с гиперпараметрами и TensorBoard

In [25]:
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_dataset = RatingsDataset(train)
val_dataset = RatingsDataset(val)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024)

criterion = nn.MSELoss()

embedding_dims = [20, 50, 100]
l2_regs = [1e-4, 1e-3, 1e-2]
epochs_list = [10, 20, 30]

log_dir = "runs/experiments"
writer = SummaryWriter(log_dir)

results = []

total_combinations = len(embedding_dims) * len(l2_regs) * len(epochs_list)
combo_counter = 0

for emb_dim in embedding_dims:
    for reg in l2_regs:
        for epochs in epochs_list:
            combo_counter += 1
            combo_desc = f"Combo {combo_counter}/{total_combinations}: dim={emb_dim}, reg={reg:.0e}, epochs={epochs}"
            print(f"\n--- {combo_desc} ---")
            
            model = MatrixFactorization(num_users, num_movies, embedding_dim=emb_dim, add_bias=True).to(device)
            optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=reg)

            train_losses = []
            val_rmses = []

            # Один прогресс-бар на все эпохи текущей комбинации
            pbar = tqdm(range(epochs), desc=f"Training {combo_desc}", unit="epoch")
            for epoch in pbar:
                train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
                val_rmse = evaluate_rmse(model, val_loader, criterion, device)
                train_losses.append(train_loss)
                val_rmses.append(val_rmse)

                tag = f"dim={emb_dim}_reg={reg}_epochs={epochs}"
                writer.add_scalar(f"Loss/train_{tag}", train_loss, epoch)
                writer.add_scalar(f"RMSE/val_{tag}", val_rmse, epoch)
                
                # Обновляем постфикс с текущими значениями
                pbar.set_postfix(loss=f"{train_loss:.4f}", rmse=f"{val_rmse:.4f}")

            results.append({
                'dim': emb_dim,
                'reg': reg,
                'epochs': epochs,
                'final_train_loss': train_losses[-1],
                'final_val_rmse': val_rmses[-1]
            })

writer.close()
print("\nЭксперименты завершены. Запустите TensorBoard: %tensorboard --logdir runs")

Device: cuda

--- Combo 1/27: dim=20, reg=1e-04, epochs=10 ---


Training Combo 1/27: dim=20, reg=1e-04, epochs=10:   0%|          | 0/10 [00:00<?, ?epoch/s]


--- Combo 2/27: dim=20, reg=1e-04, epochs=20 ---


Training Combo 2/27: dim=20, reg=1e-04, epochs=20:   0%|          | 0/20 [00:00<?, ?epoch/s]


--- Combo 3/27: dim=20, reg=1e-04, epochs=30 ---


...


--- Combo 26/27: dim=100, reg=1e-02, epochs=20 ---


Training Combo 26/27: dim=100, reg=1e-02, epochs=20:   0%|          | 0/20 [00:00<?, ?epoch/s]


--- Combo 27/27: dim=100, reg=1e-02, epochs=30 ---


Training Combo 27/27: dim=100, reg=1e-02, epochs=30:   0%|          | 0/30 [00:00<?, ?epoch/s]


Эксперименты завершены. Запустите TensorBoard: %tensorboard --logdir runs


## 7. Выбор лучшей модели и оценка на тесте

In [26]:
results_df = pd.DataFrame(results)
best_idx = results_df['final_val_rmse'].idxmin()
best_params = results_df.iloc[best_idx]
print("Лучшие параметры:")
print(best_params)

# Обучаем финальную модель
best_model = MatrixFactorization(num_users, num_movies, 
                                 embedding_dim=int(best_params['dim']), 
                                 add_bias=True).to(device)
optimizer = optim.Adam(best_model.parameters(), lr=0.005, weight_decay=best_params['reg'])

test_dataset = RatingsDataset(test)
test_loader = DataLoader(test_dataset, batch_size=1024)

for epoch in range(int(best_params['epochs'])):
    train_epoch(best_model, train_loader, optimizer, criterion, device)

test_rmse = evaluate_rmse(best_model, test_loader, criterion, device)
print(f"\nTest RMSE лучшей модели: {test_rmse:.4f}")

Лучшие параметры:
dim                 50.000000
reg                  0.000100
epochs              30.000000
final_train_loss     0.791768
final_val_rmse       0.896287
Name: 11, dtype: float64

Test RMSE лучшей модели: 0.8955


## 8. Метрики ранжирования (Precision@K, Recall@K, MAP, NDCG)

In [28]:
def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.

def ndcg_at_k(r, k):
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.
    return dcg_at_k(r, k) / dcg_max

def compute_ranking_metrics(model, test_df, train_df, num_items, k=10):
    model.eval()
    # Словари для быстрого доступа
    train_user_items = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    test_user_items = test_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    precisions = []
    recalls = []
    maps = []
    ndcgs = []

    all_item_ids = torch.arange(num_items).to(device)

    with torch.no_grad():
        for u in test_user_items.keys():
            if u not in train_user_items:
                continue
            seen = train_user_items[u]
            user_tensor = torch.full((num_items,), u, dtype=torch.long).to(device)
            scores = model(user_tensor, all_item_ids).cpu().numpy()

            # Исключаем просмотренное
            unseen_mask = np.array([i not in seen for i in range(num_items)])
            unseen_scores = scores[unseen_mask]
            unseen_items = all_item_ids.cpu().numpy()[unseen_mask]

            top_k_idx = np.argsort(unseen_scores)[-k:][::-1]
            recommended = unseen_items[top_k_idx]

            relevant = test_user_items[u]
            if not relevant:
                continue

            hits = [1 if item in relevant else 0 for item in recommended]
            precisions.append(np.sum(hits) / k)
            recalls.append(np.sum(hits) / len(relevant))

            # Average Precision
            ap = 0
            num_hits = 0
            for i, hit in enumerate(hits):
                if hit:
                    num_hits += 1
                    ap += num_hits / (i + 1)
            maps.append(ap / min(len(relevant), k) if relevant else 0)

            # NDCG
            ndcgs.append(ndcg_at_k(hits, k))

    return {
        f'Precision@{k}': np.mean(precisions),
        f'Recall@{k}': np.mean(recalls),
        f'MAP@{k}': np.mean(maps),
        f'NDCG@{k}': np.mean(ndcgs)
    }

metrics_mf = compute_ranking_metrics(best_model, test, train, num_movies, k=10)
print("Latent Factor Model (Matrix Factorization):")
for m, v in metrics_mf.items():
    print(f"{m}: {v:.4f}")

Latent Factor Model (Matrix Factorization):
Precision@10: 0.0932
Recall@10: 0.0353
MAP@10: 0.0459
NDCG@10: 0.2906


## 9. Оценка базовых моделей по ранжирующим метрикам

In [29]:
# Most Popular (простая версия)
def most_popular_recommendations(train_df, k=10):
    # Топ-K по среднему рейтингу (без порога)
    mp = train_df.groupby('movie_idx')['rating'].mean().sort_values(ascending=False).head(k).index.values
    return mp

mp_rec = most_popular_recommendations(train, k=10)

def evaluate_static_recommendations(rec_list, test_df, train_df):
    test_user_items = test_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    train_user_items = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    precisions = []
    recalls = []
    ndcgs = []
    for u, rel in test_user_items.items():
        seen = train_user_items.get(u, set())
        filtered_rec = [m for m in rec_list if m not in seen][:10]
        hits = [1 if m in rel else 0 for m in filtered_rec]
        if not rel:
            continue
        precisions.append(np.sum(hits) / len(filtered_rec) if filtered_rec else 0)
        recalls.append(np.sum(hits) / len(rel))
        ndcgs.append(ndcg_at_k(hits, len(filtered_rec)))
    return {
        'Precision@10': np.mean(precisions),
        'Recall@10': np.mean(recalls),
        'NDCG@10': np.mean(ndcgs)
    }

mp_metrics = evaluate_static_recommendations(mp_rec, test, train)
print("Most Popular:")
for k, v in mp_metrics.items():
    print(f"{k}: {v:.4f}")

Most Popular:
Precision@10: 0.0482
Recall@10: 0.0170
NDCG@10: 0.1316


In [32]:
# User/Item KNN ранжирующие метрики (ручная реализация)
def knn_ranking_metrics(predict_func, train_matrix, sim_matrix, test_df, train_df, n_items, k=10):
    test_user_items = test_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    train_user_items = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    precisions = []
    recalls = []
    ndcgs = []
    for u in test_user_items.keys():
        if u not in train_user_items:
            continue
        seen = train_user_items[u]
        # Предсказываем рейтинги для всех непросмотренных фильмов
        scores = []
        items = []
        for i in range(n_items):
            if i not in seen:
                pred = predict_func(u, i, train_matrix, sim_matrix, k=20)
                scores.append(pred)
                items.append(i)
        top_k_idx = np.argsort(scores)[-k:][::-1]
        recommended = [items[idx] for idx in top_k_idx]
        rel = test_user_items[u]
        if not rel:
            continue
        hits = [1 if m in rel else 0 for m in recommended]
        precisions.append(np.sum(hits) / k)
        recalls.append(np.sum(hits) / len(rel))
        ndcgs.append(ndcg_at_k(hits, k))
    return {
        f'Precision@{k}': np.mean(precisions),
        f'Recall@{k}': np.mean(recalls),
        f'NDCG@{k}': np.mean(ndcgs)
    }

print("Вычисление UserKNN метрик...")
ubcf_metrics = knn_ranking_metrics(user_based_predict, train_matrix, user_sim, test, train, num_movies, k=10)
print("User-based CF:")
for k, v in ubcf_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nВычисление ItemKNN метрик...")
ibcf_metrics = knn_ranking_metrics(item_based_predict, train_matrix, item_sim, test, train, num_movies, k=10)
print("Item-based CF:")
for k, v in ibcf_metrics.items():
    print(f"{k}: {v:.4f}")

Вычисление UserKNN метрик...
User-based CF:
Precision@10: 0.0819
Recall@10: 0.0307
NDCG@10: 0.2752

Вычисление ItemKNN метрик...
Item-based CF:
Precision@10: 0.0420
Recall@10: 0.0085
NDCG@10: 0.1262


## 10. Анализ Coverage и Diversity

In [33]:
def compute_coverage_diversity(model, train_df, num_items, num_users_sample=500, k=10):
    model.eval()
    train_user_items = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    all_recs = set()
    all_rec_lists = []
    sampled_users = np.random.choice(list(train_user_items.keys()), 
                                     size=min(num_users_sample, len(train_user_items)), 
                                     replace=False)
    with torch.no_grad():
        for u in sampled_users:
            seen = train_user_items[u]
            user_tensor = torch.full((num_items,), u, dtype=torch.long).to(device)
            scores = model(user_tensor, torch.arange(num_items).to(device)).cpu().numpy()
            unseen_mask = np.array([i not in seen for i in range(num_items)])
            top_k = np.argsort(scores[unseen_mask])[-k:][::-1]
            recs = np.arange(num_items)[unseen_mask][top_k]
            all_recs.update(recs)
            all_rec_lists.append(set(recs))

    coverage = len(all_recs) / num_items
    # Diversity = 1 - среднее попарное сходство (Jaccard)
    n = len(all_rec_lists)
    diversity = 0
    if n > 1:
        for i in range(n):
            for j in range(i+1, n):
                inter = len(all_rec_lists[i] & all_rec_lists[j])
                union = len(all_rec_lists[i] | all_rec_lists[j])
                diversity += 1 - inter / union if union > 0 else 1
        diversity /= (n * (n-1) / 2)
    return coverage, diversity

cov, div = compute_coverage_diversity(best_model, train, num_movies)
print(f"Coverage (доля фильмов в рекомендациях): {cov:.4f}")
print(f"Diversity (межпользовательское разнообразие): {div:.4f}")

# Сравнение с Most Popular
mp_rec_set = set(mp_rec)
cov_mp = len(mp_rec_set) / num_movies
print(f"Coverage Most Popular: {cov_mp:.4f}")

Coverage (доля фильмов в рекомендациях): 0.0184
Diversity (межпользовательское разнообразие): 0.5132
Coverage Most Popular: 0.0033


## 11. Запуск TensorBoard

В отдельной ячейке или в терминале выполните:

```
%load_ext tensorboard
%tensorboard --logdir runs
```

Или в терминале:
```
tensorboard --logdir runs
```

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir runs

## 12. Итоговое сравнение всех моделей

In [35]:
comparison = pd.DataFrame({
    'Model': ['Most Popular', 'User-based CF', 'Item-based CF', 'Matrix Factorization'],
    'RMSE': ['N/A', rmse_ubcf, rmse_ibcf, test_rmse],
    'Precision@10': [mp_metrics['Precision@10'], ubcf_metrics['Precision@10'], ibcf_metrics['Precision@10'], metrics_mf['Precision@10']],
    'Recall@10': [mp_metrics['Recall@10'], ubcf_metrics['Recall@10'], ibcf_metrics['Recall@10'], metrics_mf['Recall@10']],
    'NDCG@10': [mp_metrics['NDCG@10'], ubcf_metrics['NDCG@10'], ibcf_metrics['NDCG@10'], metrics_mf['NDCG@10']],
    'Coverage': [cov_mp, 'N/A', 'N/A', cov]
})
comparison

,Model,RMSE,Precision@10,Recall@10,NDCG@10,Coverage
0,Most Popular,N/A,0.048195,0.017018,0.131609,0.003286
1,User-based CF,0.967767,0.081937,0.030694,0.275207,N/A
2,Item-based CF,0.939772,0.041954,0.008479,0.126200,N/A
3,Matrix Factorization,0.895505,0.093195,0.035341,0.290631,0.018403


## Заключение

В ходе выполнения работы:
- Реализована модель матричного разложения с bias и регуляризацией.
- Проведён анализ влияния размерности латентного пространства, регуляризации и числа эпох (результаты в TensorBoard).
- Вычислены метрики точности (RMSE) и ранжирования (Precision@K, Recall@K, MAP, NDCG).
- Дополнительно оценено покрытие (coverage) и межпользовательское разнообразие.
- Проведено сравнение с базовыми методами.

Лучшие результаты показала модель матричного разложения, превосходя коллаборативную фильтрацию по всем метрикам.